# CLEWS: поиск похожих музыкальных фрагментов

Отдельный исследовательский ноутбук: одна песня → поиск по загруженной коллекции.
Сайт, ключи API и обучение модели не требуются. Встроенного каталога песен нет.

**Начало:** «Среда выполнения → Сменить среду выполнения → GPU (предпочтительно T4)».
Запускайте ячейки по порядку кнопками ▶. Ячейки загрузки ожидают ваш выбор файлов.

- Результат — ранжирование кандидатов и таймкоды, а не вероятность плагиата.
- Окно — длительность из checkpoint (обычно 20 с), шаг по умолчанию 5 с.
- Заменённые ноты не гарантированно распознаются. Короткие записи и тишина могут быть отклонены.
- Архив официальных весов — около **8,7 GB**; скачивается автоматически. Нужен запас диска.
- Для первой попытки возьмите одну исследуемую песню и 2–5 записей для сравнения.
- Загружаемые записи обрабатываются в вашей среде Google Colab.

Код модели: [Sony CLEWS](https://github.com/sony/clews), MIT.
Веса: [Zenodo 15045900](https://zenodo.org/records/15045900).
Источник модели закреплён на commit `517de18676e08799b51c6597a0f2af61f7a70799`.
Лицензия кода не определяет условия использования исходных записей и весов.

Ноутбук подготовлен без запуска модели или проверки в Colab.


## 1. Где хранить файлы

`USE_GOOGLE_DRIVE = True` сохраняет выбранные аудиозаписи, веса, векторы и результаты
в `Мой диск/CLEWS_demo`. При подключении Drive Google попросит доступ.
При `False` всё хранится во временной машине Colab и исчезнет после её удаления.
Не используйте общую папку, доступную посторонним.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, tempfile, zipfile

USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}
HOP_SECONDS = 5.0  # @param {type:"number"}
BATCH_SIZE = 1  # @param {type:"integer"}
MODEL_VARIANT = "dvi-clews"
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/CLEWS_demo')
else:
    ROOT = Path('/content/CLEWS_demo')
WORK = Path('/content/clews_runtime')
for path in (ROOT, WORK, ROOT/'query', ROOT/'references', ROOT/'cache', ROOT/'results', ROOT/'model'):
    path.mkdir(parents=True, exist_ok=True)
print('Данные:', ROOT)
print('Шаг:', HOP_SECONDS, 'с. Размер пакета:', BATCH_SIZE)


## 2. Установка изолированного окружения

Ядро Colab не переустанавливается. Для CLEWS создаётся отдельное окружение Python 3.11
с PyTorch 2.3.1 / CUDA 12.1 и зависимостями исходного проекта.
Установка и скачивание могут занять время. Не нужно перезапускать ядро после этой ячейки.


In [ ]:
def run(command, **kwargs):
    print('>', ' '.join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), check=True, **kwargs)

run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.9.5'])
UV = [sys.executable, '-m', 'uv']
PYTHON = WORK/'venv/bin/python'
if not PYTHON.exists():
    run(UV + ['python', 'install', '3.11'])
    run(UV + ['venv', '--python', '3.11', WORK/'venv'])
run(UV + ['pip', 'install', '--python', PYTHON,
          'torch==2.3.1', 'torchaudio==2.3.1', 'torchvision==0.18.1',
          '--index-url', 'https://download.pytorch.org/whl/cu121'])
run(UV + ['pip', 'install', '--python', PYTHON,
          'numpy==1.26.4', 'lightning==2.3.0', 'tensorboard==2.17.0',
          'einops==0.8.0', 'torchinfo==1.8.0', 'omegaconf==2.3.0',
          'tqdm==4.66.4', 'joblib==1.4.2', 'soundfile==0.12.1',
          'soxr==0.3.7', 'nnAudio==0.3.3', 'julius==0.2.7'])
if not shutil.which('ffmpeg'):
    run(['apt-get', 'update', '-qq'])
    run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'])
REPO = WORK/'clews'
COMMIT = '517de18676e08799b51c6597a0f2af61f7a70799'
if not (REPO/'.git').exists():
    run(['git', 'clone', 'https://github.com/sony/clews.git', REPO])
run(['git', '-C', REPO, 'checkout', '--detach', COMMIT])
print('Окружение установлено. Модель пока не запускалась.')


## 3. Получение официальных весов

Скачивается архив `clews.zip` с Zenodo, проверяется его опубликованная MD5,
затем извлекаются только `checkpoint_best.ckpt` и `configuration.yaml` варианта DVI.
При сохранённых весах повторное скачивание пропускается. Временный архив удаляется
после извлечения. Это **веса модели, а не база песен**.


In [ ]:
EXPECTED_MD5 = '1cc63438b41f7b4421ad67be7fe0cd15'
MODEL_DIR = ROOT/'model'/MODEL_VARIANT
CHECKPOINT = MODEL_DIR/'checkpoint_best.ckpt'
MODEL_MARKER = MODEL_DIR/'source.json'

def file_digest(path, algorithm='sha256'):
    digest = hashlib.new(algorithm)
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(4*1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

ready = False
if CHECKPOINT.exists() and (MODEL_DIR/'configuration.yaml').exists() and MODEL_MARKER.exists():
    marker = json.loads(MODEL_MARKER.read_text())
    ready = (marker.get('archive_md5') == EXPECTED_MD5
             and marker.get('checkpoint_sha256') == file_digest(CHECKPOINT)
             and marker.get('config_sha256') == file_digest(MODEL_DIR/'configuration.yaml'))
if not ready:
    archive = WORK/'clews.zip'
    run(['curl', '--fail', '--location', '--retry', '3', '--continue-at', '-',
         'https://zenodo.org/records/15045900/files/clews.zip?download=1', '-o', archive])
    if file_digest(archive, 'md5') != EXPECTED_MD5:
        archive.unlink(missing_ok=True)
        raise RuntimeError('Архив не прошёл MD5. Повторите эту ячейку для нового скачивания.')
    with zipfile.ZipFile(archive) as source:
        candidates = [n for n in source.namelist()
                      if Path(n).name == 'checkpoint_best.ckpt' and MODEL_VARIANT in n]
        if len(candidates) != 1:
            raise RuntimeError('Неоднозначный состав архива. Найдены checkpoint: ' +
                               repr([n for n in source.namelist() if n.endswith('.ckpt')]))
        checkpoint_member = candidates[0]
        config_member = str(Path(checkpoint_member).parent/'configuration.yaml')
        if config_member not in source.namelist():
            raise RuntimeError('В архиве отсутствует configuration.yaml рядом с весами.')
        MODEL_DIR.mkdir(parents=True, exist_ok=True)
        for member, target in ((checkpoint_member, CHECKPOINT),
                               (config_member, MODEL_DIR/'configuration.yaml')):
            temporary = target.with_suffix(target.suffix+'.partial')
            with source.open(member) as inp, open(temporary, 'wb') as out:
                shutil.copyfileobj(inp, out, length=4*1024*1024)
            os.replace(temporary, target)
    MODEL_MARKER.write_text(json.dumps(dict(
        archive_md5=EXPECTED_MD5, checkpoint_sha256=file_digest(CHECKPOINT),
        config_sha256=file_digest(MODEL_DIR/'configuration.yaml'),
        source='https://zenodo.org/records/15045900', variant=MODEL_VARIANT), indent=2))
    archive.unlink()
print('Веса готовы:', CHECKPOINT)


## 4. Подготовка программы поиска

Эта ячейка записывает самостоятельную программу, встроенную в ноутбук.
Она вызывает исходную модель CLEWS небольшими пакетами окон и сохраняет векторы
в NPZ. Кэш привязан к содержимому аудио, весам, конфигурации и параметрам окон.


In [ ]:
HELPER_SOURCE = '"""Standalone CLEWS fragment search used by the accompanying Colab notebook.\n\nThe upstream model is downloaded separately; no website imports are used.\n"""\nimport argparse\nimport csv\nimport hashlib\nimport importlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport tempfile\n\nimport numpy as np\nimport torch\nfrom omegaconf import OmegaConf\n\nSCHEMA = "melawdy-clews-colab-v1"\nREPO_COMMIT = "517de18676e08799b51c6597a0f2af61f7a70799"\nEXTENSIONS = {".mp3", ".wav", ".flac", ".ogg", ".m4a", ".aac"}\nFIELDS = ["reference", "distance", "query_start", "query_end", "reference_start", "reference_end"]\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with open(path, "rb") as stream:\n        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef decode(path, sr, max_seconds):\n    with tempfile.TemporaryFile() as output:\n        proc = subprocess.run([\n            "ffmpeg", "-v", "error", "-nostdin", "-i", str(path),\n            "-t", str(max_seconds + 1), "-vn", "-ac", "1", "-ar", str(sr),\n            "-f", "f32le", "pipe:1",\n        ], stdout=output, stderr=subprocess.PIPE, timeout=180)\n        if proc.returncode:\n            raise ValueError(proc.stderr.decode(errors="replace")[-1500:])\n        output.seek(0)\n        audio = np.frombuffer(output.read(), dtype="<f4").copy()\n    if len(audio) > max_seconds * sr:\n        raise ValueError(f"Запись длиннее {max_seconds} секунд. Загрузите более короткий файл.")\n    if not len(audio) or not np.isfinite(audio).all():\n        raise ValueError("Пустое или повреждённое аудио.")\n    return audio\n\n\ndef load_model(repo, checkpoint, device):\n    sys.path.insert(0, str(repo))\n    conf = OmegaConf.load(checkpoint.parent / "configuration.yaml")\n    if conf.model.name != "clews":\n        raise ValueError("Выбран checkpoint другой архитектуры; требуется CLEWS.")\n    module = importlib.import_module("models.clews")\n    model = module.Model(conf.model, sr=int(conf.data.samplerate))\n    # Only the checksum-verified official Zenodo checkpoint is used by the notebook.\n    state = torch.load(checkpoint, map_location="cpu", weights_only=False)\n    model.load_state_dict(state["model"], strict=True)\n    del state\n    model.eval().to(device)\n    return model, conf\n\n\ndef extract(path, model, cache_dir, model_hash, config_hash, window, hop, batch_size, device, max_seconds):\n    audio_hash = sha256(path)\n    identity = dict(schema=SCHEMA, audio_sha256=audio_hash, model_sha256=model_hash,\n                    repo_commit=REPO_COMMIT, configuration_sha256=config_hash, window=window, hop=hop,\n                    samplerate=model.sr, min_rms=0.0001, max_seconds=max_seconds)\n    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()\n    cache_path = cache_dir / (key + ".npz")\n    if cache_path.exists():\n        with np.load(cache_path, allow_pickle=False) as saved:\n            return dict(path=str(path), sha256=audio_hash, duration=float(saved["duration"]),\n                        starts=saved["starts"], vectors=saved["vectors"], cache=str(cache_path))\n    audio = decode(path, model.sr, max_seconds)\n    nwin = int(round(window * model.sr))\n    nhop = int(round(hop * model.sr))\n    if len(audio) < nwin:\n        raise ValueError(f"Нужно не менее {window:g} секунд аудио; короткие записи не дополняются повторением.")\n    positions = list(range(0, len(audio) - nwin + 1, nhop))\n    if positions[-1] != len(audio) - nwin:\n        positions.append(len(audio) - nwin)\n    positions = [p for p in positions if np.sqrt(np.mean(audio[p:p+nwin].astype(np.float64)**2)) >= 0.0001]\n    if not positions:\n        raise ValueError("Все окна слишком тихие: анализ не выполнен.")\n    vectors = []\n    with torch.inference_mode():\n        for offset in range(0, len(positions), batch_size):\n            chunk = positions[offset:offset + batch_size]\n            samples = np.stack([audio[p:p+nwin] for p in chunk])\n            z = model(torch.from_numpy(samples).to(device), shingle_len=window, shingle_hop=window)\n            z = z[:, 0, :].float().cpu().numpy()\n            if not np.isfinite(z).all():\n                raise ValueError("Модель вернула некорректные векторы.")\n            vectors.append(z)\n            print(f"  {path.name}: {min(offset + batch_size, len(positions))}/{len(positions)} окон", flush=True)\n    result = dict(path=str(path), sha256=audio_hash, duration=len(audio)/model.sr,\n                  starts=np.asarray(positions, dtype=np.float64)/model.sr,\n                  vectors=np.concatenate(vectors), cache=str(cache_path))\n    temporary = cache_path.with_suffix(".tmp.npz")\n    np.savez_compressed(temporary, vectors=result["vectors"], starts=result["starts"], duration=result["duration"])\n    os.replace(temporary, cache_path)\n    return result\n\n\ndef find_matches(query, ref, window, count):\n    # Same segment metric as CLEWS: squared Euclidean distance / embedding dimension.\n    q = torch.from_numpy(query["vectors"])\n    r = torch.from_numpy(ref["vectors"])\n    from lib.tensor_ops import pairwise_distance_matrix\n    distances = pairwise_distance_matrix(q, r, mode="nsqeuc").numpy()\n    available = distances.copy()\n    matches = []\n    for _ in range(count):\n        index = np.unravel_index(np.argmin(available), available.shape)\n        score = float(available[index])\n        if not math.isfinite(score):\n            break\n        qs, rs = float(query["starts"][index[0]]), float(ref["starts"][index[1]])\n        matches.append(dict(reference=ref["path"], distance=score,\n                            query_start=qs, query_end=qs+window,\n                            reference_start=rs, reference_end=rs+window))\n        # Suppress near-duplicate windows in both recordings, retaining separate repetitions.\n        qm = np.abs(query["starts"] - qs) < window * 0.75\n        rm = np.abs(ref["starts"] - rs) < window * 0.75\n        available[np.ix_(qm, rm)] = np.inf\n    return matches, distances\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    for name in ("repo", "checkpoint", "query", "references", "cache", "output"):\n        parser.add_argument("--" + name, type=Path, required=True)\n    parser.add_argument("--hop", type=float, default=5)\n    parser.add_argument("--batch-size", type=int, default=1)\n    parser.add_argument("--matches-per-track", type=int, default=3)\n    parser.add_argument("--max-seconds", type=int, default=1800)\n    args = parser.parse_args()\n    if args.hop <= 0 or args.batch_size < 1 or args.matches_per_track < 1 or args.max_seconds < 20:\n        parser.error("Некорректные параметры поиска.")\n    args.cache.mkdir(parents=True, exist_ok=True)\n    args.output.mkdir(parents=True, exist_ok=True)\n    if not args.query.is_file():\n        raise ValueError("Выберите исследуемый файл в ячейке загрузки.")\n    refs = sorted(p for p in args.references.rglob("*") if p.is_file() and p.suffix.lower() in EXTENSIONS)\n    if not refs:\n        raise ValueError("Загрузите хотя бы одну запись для сравнения.")\n    torch.manual_seed(43)\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    print("Устройство:", device, flush=True)\n    if device == "cpu":\n        print("GPU отсутствует; обработка на CPU может быть медленной.", flush=True)\n    model_hash = sha256(args.checkpoint)\n    config_hash = sha256(args.checkpoint.parent / "configuration.yaml")\n    model, conf = load_model(args.repo, args.checkpoint, device)\n    window = float(conf.model.shingling.len)\n    if args.hop > window:\n        raise ValueError("Шаг больше окна создаёт пропуски; уменьшите hop.")\n    def embed(path):\n        return extract(path, model, args.cache, model_hash, config_hash, window, args.hop,\n                       args.batch_size, device, args.max_seconds)\n    query = embed(args.query)\n    matches, errors, excluded, manifest = [], [], [], []\n    best_matrix, best_score = None, float("inf")\n    for path in refs:\n        print("Сравнение:", path.name, flush=True)\n        try:\n            if sha256(path) == query["sha256"]:\n                excluded.append(dict(path=str(path), reason="Файл идентичен исследуемому по SHA-256"))\n                continue\n            ref = embed(path)\n            found, matrix = find_matches(query, ref, window, args.matches_per_track)\n            matches.extend(found)\n            manifest.append({k: ref[k] for k in ("path", "sha256", "duration", "cache")})\n            if found and found[0]["distance"] < best_score:\n                best_score = found[0]["distance"]\n                best_matrix = dict(distances=matrix, query_starts=query["starts"], reference_starts=ref["starts"])\n        except (ValueError, RuntimeError, subprocess.TimeoutExpired) as exc:\n            errors.append(dict(path=str(path), error=str(exc)))\n            print("Не обработан:", path.name, str(exc), flush=True)\n            if device == "cuda":\n                torch.cuda.empty_cache()\n    matches.sort(key=lambda m: m["distance"])\n    report = dict(schema=SCHEMA, status="ranked_candidates" if matches else "analysis_not_completed",\n                  notice="Это ранжирование сходства, не процент плагиата. Ближайший кандидат может быть несходным.",\n                  repo_commit=REPO_COMMIT, checkpoint_sha256=model_hash, configuration_sha256=config_hash, device=device,\n                  metric="squared_euclidean_div_embedding_dimension", window_seconds=window,\n                  hop_seconds=args.hop, query={k: query[k] for k in ("path", "sha256", "duration", "cache")},\n                  references=manifest, errors=errors, excluded=excluded, matches=matches)\n    (args.output / "results.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")\n    with (args.output / "matches.csv").open("w", encoding="utf-8-sig", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=FIELDS)\n        writer.writeheader()\n        writer.writerows(matches)\n    if best_matrix is not None:\n        np.savez_compressed(args.output / "best_match_matrix.npz", **best_matrix)\n    elif (args.output / "best_match_matrix.npz").exists():\n        (args.output / "best_match_matrix.npz").unlink()\n    print(f"Готово: {len(manifest)} записей обработано; {len(errors)} ошибок; {len(excluded)} исключено.", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
HELPER = WORK/'clews_search.py'
HELPER.write_text(HELPER_SOURCE, encoding='utf-8')
print('Программа поиска подготовлена.')


## 5. Исследуемая песня

Выберите **один** MP3, WAV, FLAC, OGG, M4A или AAC длиной от 20 секунд до 30 минут.
Чтобы исследовать другую песню, повторите эту ячейку и ячейки 7–9.


In [ ]:
from google.colab import files
AUDIO_EXTENSIONS = {'.mp3', '.wav', '.flac', '.ogg', '.m4a', '.aac'}

def upload_audio(destination, single=False):
    # Colab also writes the uploaded files into the current working directory.
    # Keep that temporary directory separate from notebook/repository files.
    previous = os.getcwd()
    with tempfile.TemporaryDirectory(prefix='clews_upload_') as upload_dir:
        try:
            os.chdir(upload_dir)
            incoming = files.upload()
        finally:
            os.chdir(previous)
        if not incoming or (single and len(incoming) != 1):
            raise ValueError('Нужно выбрать ровно один файл.' if single else 'Файлы не выбраны.')
        if any(Path(name).suffix.lower() not in AUDIO_EXTENSIONS for name in incoming):
            raise ValueError('Поддерживаются только MP3, WAV, FLAC, OGG, M4A, AAC.')
        saved = []
        for name, data in incoming.items():
            safe_name = Path(name).name
            digest = hashlib.sha256(data).hexdigest()[:12]
            target = destination/(digest+'_'+safe_name)
            target.write_bytes(data)
            saved.append(target)
        return saved

QUERY = upload_audio(ROOT/'query', single=True)[0]
print('Исследуемая песня:', QUERY.name)


## 6. Песни для сравнения

Выберите один или несколько аудиофайлов. Они добавляются в коллекцию `references`.
Можно повторять эту ячейку, чтобы расширять коллекцию. Для большого числа файлов
удобнее вручную скопировать их в эту папку через Google Drive, если он подключён.
Файл, полностью идентичный исследуемому по SHA-256, будет исключён.

Ничего не ищется в интернете: проверяются только записи из этой папки и её подпапок.


In [ ]:
added = upload_audio(ROOT/'references')
reference_files = sorted(p for p in (ROOT/'references').rglob('*')
                         if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS)
print('Добавлено:', len(added), 'Всего записей:', len(reference_files))
for path in reference_files:
    print(path.relative_to(ROOT/'references'))


## 7. Найти похожие фрагменты

Каждый трек обрабатывается один раз при неизменных параметрах. Поиск использует
метрику CLEWS: квадрат евклидова расстояния, делённый на размерность вектора.
**Меньше — ближе**, но подтверждённого универсального порога нет.

Выводятся по три лучших пары окон для каждой записи; близкие перекрывающиеся
повторы подавляются. Это сравнение фрагментов, а не полное воспроизведение
официальной процедуры оценки песен CLEWS. Тихие окна пропускаются.
Последнее полное окно может иметь шаг меньше выбранного, чтобы покрыть конец трека.


In [ ]:
if 'QUERY' not in globals() or not QUERY.is_file():
    raise RuntimeError('Сначала выполните ячейку 5 и выберите исследуемую песню.')
if HOP_SECONDS <= 0 or BATCH_SIZE < 1:
    raise ValueError('Шаг должен быть положительным, размер пакета — не меньше 1.')
# A separate output folder prevents displaying results of an earlier failed run.
import datetime
RUN_DIR = ROOT/'results'/datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
run([PYTHON, HELPER, '--repo', REPO, '--checkpoint', CHECKPOINT,
     '--query', QUERY, '--references', ROOT/'references', '--cache', ROOT/'cache',
     '--output', RUN_DIR, '--hop', HOP_SECONDS, '--batch-size', BATCH_SIZE])
print('Результаты:', RUN_DIR)


## 8. Результаты и прослушивание

Первые строки — ближайшие кандидаты, даже если все песни фактически разные.
Слушайте найденные окна и проверяйте контекст. Ошибки чтения и исключённые записи
показываются отдельно, а не выдаются за отсутствие совпадений.


In [ ]:
from IPython.display import Audio, HTML, display
from html import escape

report = json.loads((RUN_DIR/'results.json').read_text())
MATCHES = report['matches']
def clock(seconds):
    return f'{int(seconds)//60:02d}:{seconds%60:04.1f}'

def table(rows, columns):
    headers = ''.join('<th>'+escape(label)+'</th>' for key, label in columns)
    body = ''.join('<tr>'+''.join('<td>'+escape(str(row.get(key, '')))+'</td>'
                                 for key, label in columns)+'</tr>' for row in rows)
    display(HTML('<table><thead><tr>'+headers+'</tr></thead><tbody>'+body+'</tbody></table>'))

print(report['notice'])
rows = [dict(rank=i+1, reference=Path(m['reference']).name,
             query_time=clock(m['query_start'])+'–'+clock(m['query_end']),
             reference_time=clock(m['reference_start'])+'–'+clock(m['reference_end']),
             distance=f"{m['distance']:.6f}") for i, m in enumerate(MATCHES)]
table(rows[:30], [('rank','№'), ('reference','Запись'), ('query_time','Исследуемая песня'),
                 ('reference_time','Сравниваемая песня'), ('distance','Расстояние ↓')])
for issue in report['errors']:
    print('Ошибка:', Path(issue['path']).name, issue['error'])
for issue in report['excluded']:
    print('Исключён:', Path(issue['path']).name, issue['reason'])
if not MATCHES:
    print('Нет результата ранжирования. Посмотрите ошибки, длительность файлов и исключения.')

matrix_path = RUN_DIR/'best_match_matrix.npz'
if matrix_path.exists():
    import numpy as np
    import matplotlib.pyplot as plt
    with np.load(matrix_path, allow_pickle=False) as matrix:
        fig, ax = plt.subplots(figsize=(9,4))
        mesh = ax.pcolormesh(matrix['reference_starts'], matrix['query_starts'],
                             matrix['distances'], shading='nearest', cmap='viridis_r')
        ax.set_xlabel('Начало окна сравниваемой записи, с')
        ax.set_ylabel('Начало окна исследуемой записи, с')
        ax.set_title('Ближайшая запись: '+Path(MATCHES[0]['reference']).name)
        fig.colorbar(mesh, ax=ax, label='Расстояние (меньше — ближе)')
        plt.show()


### Прослушать выбранную строку

Укажите номер из таблицы и выполните ячейку. Можно слушать и строки после 30-й:
полный список находится в JSON/CSV. Плееры воспроизводят найденные окна.


In [ ]:
MATCH_NUMBER = 1  # @param {type:"integer"}
if not MATCHES or not (1 <= MATCH_NUMBER <= len(MATCHES)):
    raise ValueError('Нет такой строки. Сначала получите результаты поиска.')
match = MATCHES[MATCH_NUMBER-1]
for label, path, start, end in (
    ('Исследуемая песня', report['query']['path'], match['query_start'], match['query_end']),
    ('Сравниваемая запись', match['reference'], match['reference_start'], match['reference_end']),
):
    clip = WORK/('listen_query.mp3' if label == 'Исследуемая песня' else 'listen_reference.mp3')
    subprocess.run(['ffmpeg', '-v', 'error', '-nostdin', '-y', '-ss', str(start), '-i', path,
                    '-t', str(end-start), '-vn', '-ac', '1', '-ar', '22050', str(clip)], check=True)
    print(label, Path(path).name, clock(start)+'–'+clock(end))
    display(Audio(filename=str(clip)))


## 9. Скачать отчёт и при необходимости векторы

ZIP содержит JSON, CSV и матрицу расстояний ближайшей записи.
При `EXPORT_EMBEDDINGS = True` добавляются использованные NPZ-векторы и манифест.
Исходное аудио и модель в ZIP не входят. Для повторного использования индекса нужно
сохранить ту же версию модели, параметры и соответствие векторов записям.


In [ ]:
EXPORT_EMBEDDINGS = False  # @param {type:"boolean"}
export_zip = WORK/'clews_results.zip'
with zipfile.ZipFile(export_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in RUN_DIR.iterdir():
        if path.is_file():
            archive.write(path, 'report/'+path.name)
    if EXPORT_EMBEDDINGS:
        entries = [report['query']] + report['references']
        manifest = []
        for entry in entries:
            cache = Path(entry['cache'])
            if cache.exists():
                archive.write(cache, 'embeddings/'+cache.name)
                manifest.append(dict(audio_path=entry['path'], audio_sha256=entry['sha256'],
                                     embeddings='embeddings/'+cache.name))
        archive.writestr('embeddings_manifest.json', json.dumps(dict(
            checkpoint_sha256=report['checkpoint_sha256'], repo_commit=report['repo_commit'],
            window_seconds=report['window_seconds'], hop_seconds=report['hop_seconds'],
            files=manifest), ensure_ascii=False, indent=2))
files.download(str(export_zip))


## Если возникла ошибка

- **Недостаточно GPU-памяти:** оставьте `BATCH_SIZE = 1`, возьмите более короткие записи.
- **CUDA недоступна:** проверьте выбор GPU. Программа допускает CPU, но он медленнее.
- **Новая GPU несовместима с PyTorch 2.3.1:** выберите T4; более новое оборудование может
  потребовать обновления PyTorch и согласованного torchaudio.
- **Архив Zenodo не скачивается:** повторите ячейку 3; скачивание умеет продолжаться.
- **Среда Colab отключилась:** повторите ячейки по порядку. При подключённом Drive
  сохранённые веса и векторы будут использованы повторно.
- **Короткий файл / тишина:** возьмите запись от 20 секунд с достаточно громкой музыкой.
- **Не узнаёт похожую мелодию:** это ограничение результата модели, не доказательство
  отсутствия заимствования. Сравните больше контекста и послушайте оба фрагмента.

[Ограничения Colab](https://research.google.com/colaboratory/faq.html).
